# Les plus grands modèles open-weight face à nos modèles internes : nom, naissance, adresse

**La question (veille et innovation) :** les plus grands modèles **open-weight** (poids publics, donc installables
un jour chez nous), bien plus gros que nos modèles internes, lisent-ils mieux nos bulletins manuscrits ? Sur trois
sujets : **nom et prénom**, **date et lieu de naissance**, **adresse du lieu de travail**.

**La méthode :** la même que les notebooks précédents. La même image pour tous (le bloc de la question, découpé par
l'étape 1), la même consigne, les lectures **mises au format du Fiqual** (consignes de saisie du PMQ), jugées avec la
**vérité arbitrée par le SeRN**, comparées par le test exact de **McNemar** à notre meilleur modèle interne.

Les modèles open-weight sont testés via **Eden AI**, qui transmet les requêtes à un hébergeur : les images sortent
de l'Insee le temps du test. Si l'un d'eux se détache, la suite serait de l'installer en interne.

## Comment lancer

1. **Cellule 1** : `TEST_RAPIDE = True`, les sujets, le plafond de dépense.
2. **Cellule 2** : le catalogue d'Eden AI liste les modèles open-weight qui lisent les images ; ceux des familles
   visées sont ajoutés automatiquement aux candidats (vérifier la liste affichée).
3. **Run All.** L'essai (3 bulletins par modèle et par sujet) écarte les noms faux sans dépense.
4. Si tout est bon : `TEST_RAPIDE = False`, Run All (rien n'est relu ni repayé).
5. **Lire** : cellule 7 (tableau croisé modèles × sujets), cellule 8 (McNemar contre l'interne), cellule 9 (coût).

| Cellule | Ce qu'elle fait |
|---|---|
| 1 | Réglages : sujets, modèles, plafond |
| 2 | Catalogue Eden AI : les open-weight qui lisent les images |
| 3 | Les lecteurs : candidats open-weight + références internes |
| 4 | Préparer chaque sujet : Fiqual, vérité, découpes |
| 5 | Essai puis lectures (cache) |
| 6 | Évaluer chaque sujet (consignes de saisie appliquées) |
| 7 | **Tableau croisé** : chaque modèle sur chaque sujet |
| 8 | **McNemar** : chaque grand modèle contre le meilleur interne |
| 9 | Taille, temps, coût |
| 10 | Pages HTML des erreurs, une par sujet |
| 11 – 12 | Excel, MLflow, phrases pour la réunion |

In [ ]:
# 1. REGLAGES
import logging
import re
import subprocess
import tempfile
from pathlib import Path

import mlflow
import pandas as pd
import requests
from IPython.display import display

from mesure_qualite import lecture as lc
from mesure_qualite.evaluation import double_mlflow as dm
from mesure_qualite.lecture import etude

SUJETS = ["identite", "naissance", "adresse_travail"]   # nom et prenom, date et lieu de naissance, adresse (Q23)
CAMPAGNE = "RG25"
TEST_RAPIDE = True                  # True d'abord : 10 bulletins par sujet ; puis False
N_RAPIDE = 10
CONSIGNES = ["V1"]                  # une consigne (detaillee) pour limiter le cout ; ["V0", "V1"] pour comparer

# ---- Candidats : les plus grands open-weight qui lisent les images, via Eden AI ----
# Noms a confirmer dans le catalogue (cellule 2) ; un nom faux est ecarte a l'essai, sans cout.
CANDIDATS_EDEN = {
    "qwen/qwen3-vl-plus":           "Qwen3-VL (famille 235B-A22B), specialise documents et formulaires",
    "qwen/qwen3.5-397b-a17b":       "Qwen3.5, 397B dont 17B actifs : le plus grand Qwen qui lit les images",
    "zai/glm-5.3-flash":            "GLM-5.3, 320B dont ~18B actifs, licence MIT (aout 2026)",
    "mistral/mistral-medium-3-5-128b": "Mistral Medium 3.5, 128B, francais",
    "moonshotai/kimi-k3-1-pro":     "Kimi K3, 2 800B dont 104B actifs (licence propre)",
}
AJOUT_AUTO_CATALOGUE = True         # ajoute les open-weight du catalogue qui lisent les images (familles ci-dessous)
FAMILLES_OPEN_WEIGHT = r"qwen|glm|kimi|mistral|pixtral|internvl|gemma|llama|deepseek|molmo|minimax|step|nemotron|apertus"
SUFFIXE_EU = ""                     # "@eu" = calcul garde dans l'Union europeenne quand Eden AI le propose

# ---- References internes (SSP Cloud, gratuites, aucune image ne sort) ----
REFERENCES_INTERNES = ["gemma4-26b-moe", "qwen3-8-27b"]

# ---- Tailles connues (milliards de parametres : total / actifs), pour le tableau de la cellule 9 ----
TAILLES = {
    "gemma4-26b-moe": (26, 4), "qwen3-8-27b": (27, 27), "qwen3-6-35b-moe": (35, 3),
    "qwen3.5-397b-a17b": (397, 17), "qwen3-vl-plus": (235, 22), "glm-5.3-flash": (320, 18),
    "mistral-medium-3-5-128b": (128, 128), "kimi-k3-1-pro": (2800, 104),
}

# ---- Garde-fous ----
PLAFOND_DOLLARS = 40.0
DUREE_MAX_MINUTES = 60
DOCUMENTS_PAR_AN = 15 * 2450
COLONNES_FIQUAL = {"identite": None, "naissance": None, "adresse_travail": None}   # ex. {"PAYS": 22}

RACINE = None                       # None = le bucket du projet
LECTURES = Path.home() / "work" / "banc-htr" / "resultats" / "lecture"
LECTURES.mkdir(parents=True, exist_ok=True)
SUFFIXE = "_rapide" if TEST_RAPIDE else ""
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
print(f"{CAMPAGNE} | {'TEST RAPIDE' if TEST_RAPIDE else 'COMPLET'} | sujets {SUJETS} | plafond {PLAFOND_DOLLARS} $")

In [ ]:
# 2. CATALOGUE EDEN AI : quels modeles open-weight lisent les images ?
lc.cle_api("eden")
r = requests.get("https://api.edenai.run/v3/models",
                 headers={"Authorization": f"Bearer {lc.cle_api('eden')}"}, timeout=60)
r.raise_for_status()
j = r.json()
CATALOGUE = pd.json_normalize(j.get("data", j.get("models", j)) if isinstance(j, dict) else j)
texte = CATALOGUE.astype(str).agg(" ".join, axis=1).str.lower()
col_id = next((c for c in ["id", "model", "name", "slug"] if c in CATALOGUE.columns), CATALOGUE.columns[0])
vision = texte.str.contains(r"image|vision|multimodal|\bvl\b|-vl")
ouverts = texte.str.contains(FAMILLES_OPEN_WEIGHT)
OPEN_VISION = CATALOGUE[vision & ouverts]
print(f"{len(CATALOGUE)} modeles au catalogue | {int(vision.sum())} lisent les images | "
      f"{len(OPEN_VISION)} open-weight parmi eux")
with pd.option_context("display.max_colwidth", 60):
    display(OPEN_VISION.iloc[:, :8])
if AJOUT_AUTO_CATALOGUE:
    for m in OPEN_VISION[col_id].astype(str):
        CANDIDATS_EDEN.setdefault(m, "trouve dans le catalogue Eden AI (open-weight, images)")
print(f"Candidats retenus pour l'essai : {list(CANDIDATS_EDEN)}")

In [ ]:
# 3. LES LECTEURS : candidats open-weight (Eden AI) + references internes (SSP Cloud)
lc.cle_api("interne")
LECTEURS = [lc.Lecteur(m, "interne") for m in REFERENCES_INTERNES]
LECTEURS += [lc.Lecteur(m + SUFFIXE_EU if SUFFIXE_EU and not m.endswith(SUFFIXE_EU) else m, "eden")
             for m in CANDIDATS_EDEN]
INTERNES = [l.nom for l in LECTEURS if not l.externe]
COMPTEURS = lc.Compteurs()
display(pd.DataFrame([{"lecteur": l.nom, "fournisseur": l.fournisseur,
                       "pourquoi": CANDIDATS_EDEN.get(l.modele.removesuffix(SUFFIXE_EU), "reference interne")}
                      for l in LECTEURS]))

In [ ]:
# 4. PREPARER CHAQUE SUJET : Fiqual, verite arbitree, decoupe du bloc sur chaque bulletin (etape 1)
D = {}
for s in SUJETS:
    D[s] = etude.preparer(s, CAMPAGNE, racine=RACINE, colonnes=COLONNES_FIQUAL.get(s),
                          limite=N_RAPIDE if TEST_RAPIDE else None)
    display(D[s].verite.groupby(["champ", "groupe"]).size().unstack(fill_value=0)
            .reindex(columns=["A", "B", "C"], fill_value=0))

In [ ]:
# 5. ESSAI PUIS LECTURES : chaque modele lit 3 bulletins par sujet ; ceux qui repondent au bon format lisent tout.
#    Cache : rien n'est relu ni repaye. Plafond de depense commun a tous les sujets.
L = {}
for s in SUJETS:
    print(f"\n===== {D[s].sujet.titre} =====")
    L[s] = etude.lire_tous(D[s], LECTEURS, CONSIGNES, LECTURES, COMPTEURS,
                           plafond_dollars=PLAFOND_DOLLARS, duree_max_minutes=DUREE_MAX_MINUTES)
    display(L[s].essai[["lecteur", "lus", "json_ok", "s_par_appel", "version_servie",
                        "cout_estime_tout_$", "retenu", "erreur"]])
print(f"\nCout cumule Eden AI : {COMPTEURS.cout:.2f} $ | parametres refuses : {COMPTEURS.adaptations or 'aucun'}")

In [ ]:
# 6. EVALUER CHAQUE SUJET : lectures mises au format du Fiqual (consignes de saisie), puis jugees
#    contre la verite arbitree, sur les memes lignes pour tous.
EV = {}
for s in SUJETS:
    EV[s] = etude.evaluer(D[s], L[s].lectures)
    print(f"\n===== {D[s].sujet.titre} : {EV[s].principal} =====")
    display(EV[s].vue[["lecteur", "lignes", "justes_%", "IC95", "avouees_%", "silencieuses_%"]])
    if EV[s].ecartes:
        print(f"Ecartes (moins de 95 % des bulletins lus) : {EV[s].ecartes}")

In [ ]:
# 7. TABLEAU CROISE : chaque modele (sa meilleure consigne) sur chaque sujet, champ principal
lignes = []
for s in SUJETS:
    v = EV[s].vue.set_index("lecteur")
    for modele, nom in EV[s].meilleure.items():
        lignes.append({"modele": modele, "sujet": D[s].sujet.titre, "justes_%": v.loc[nom, "justes_%"],
                       "silencieuses_%": v.loc[nom, "silencieuses_%"]})
    lignes.append({"modele": "PMQ (saisie humaine)", "sujet": D[s].sujet.titre,
                   "justes_%": v.loc["PMQ", "justes_%"], "silencieuses_%": v.loc["PMQ", "silencieuses_%"]})
CROISE = pd.DataFrame(lignes)
TAB_JUSTES = CROISE.pivot(index="modele", columns="sujet", values="justes_%")
TAB_JUSTES["moyenne"] = TAB_JUSTES.mean(axis=1).round(1)
TAB_JUSTES = TAB_JUSTES.sort_values("moyenne", ascending=False)
print("Justes % (champ principal de chaque sujet, verite arbitree) :")
display(TAB_JUSTES)
print("Erreurs silencieuses % :")
display(CROISE.pivot(index="modele", columns="sujet", values="silencieuses_%").loc[TAB_JUSTES.index])

In [ ]:
# 8. McNEMAR : chaque grand modele contre le MEILLEUR modele interne, sujet par sujet (memes bulletins)
DUELS = []
for s in SUJETS:
    ev = EV[s]
    j = ev.vue.set_index("lecteur")["justes_%"]
    internes = [ev.meilleure[m] for m in INTERNES if m in ev.meilleure]
    if not internes:
        continue
    ref = max(internes, key=lambda n: j[n])
    for modele, nom in ev.meilleure.items():
        if nom in internes:
            continue
        d_ = etude.duel(ev, nom, ref)
        DUELS.append({"sujet": D[s].sujet.titre, "grand modele": nom, "justes_%": j[nom],
                      "meilleur interne": ref, "interne justes_%": j[ref],
                      "lui seul juste": d_["a seul juste"], "interne seul juste": d_["b seul juste"],
                      "p": round(d_["p"], 4),
                      "conclusion": ("meilleur que l'interne" if d_["p"] < 0.05 and d_["a seul juste"] > d_["b seul juste"]
                                     else "moins bon que l'interne" if d_["p"] < 0.05 else "pas d'ecart etabli")})
DUELS = pd.DataFrame(DUELS)
display(DUELS)

In [ ]:
# 9. TAILLE, TEMPS, COUT : par modele, tous sujets confondus
COUTS = []
for l in LECTEURS:
    sec, cout = [], []
    for s in SUJETS:
        for nom, (lecteur, _, lus) in L[s].lectures.items():
            if lecteur == l:
                sec += list(pd.to_numeric(lus.secondes, errors="coerce").dropna())
                cout += list(pd.to_numeric(lus.cout, errors="coerce").dropna())
    if not sec:
        continue
    court = l.modele.split("/")[-1].removesuffix(SUFFIXE_EU)
    tot, act = TAILLES.get(court, (None, None))
    par_doc = (sum(cout) / len(cout)) if cout else (0.0 if not l.externe else None)
    COUTS.append({"modele": l.nom, "taille totale (Md)": tot, "actifs par jeton (Md)": act,
                  "s par bloc": round(sum(sec) / len(sec), 2),
                  "$ pour 1000 blocs": round(par_doc * 1000, 2) if par_doc is not None else None,
                  "$ par an (1 sujet)": round(par_doc * DOCUMENTS_PAR_AN) if par_doc is not None else None,
                  "images hors Insee": "oui" if l.externe else "non"})
COUTS = pd.DataFrame(COUTS)
display(COUTS)
print("Tailles : a verifier sur la fiche de chaque modele ; « actifs » = parametres utilises par jeton (MoE).")

In [ ]:
# 10. PAGES HTML DES ERREURS : une par sujet (comparaison avec la verite, puis chaque bulletin en erreur)
PAGES = {}
for s in SUJETS:
    ev = EV[s]
    noms = list(ev.meilleure.values())
    PAGES[s] = lc.galerie_erreurs(ev.table, noms, D[s].images, D[s].sujet,
                                  titre=f"campagne {CAMPAGNE}, {len(D[s].images)} bulletins, grands modeles open-weight")
    f = LECTURES / f"erreurs_grands_modeles_{s}_{CAMPAGNE}{SUFFIXE}.html"
    f.write_text(PAGES[s], encoding="utf-8")
    print(f"{D[s].sujet.titre} : {f} ({len(PAGES[s]) / 1e6:.1f} Mo)")

In [ ]:
# 11. EXCEL ET MLflow : tout le detail, serveur du projet et serveur personnel
SERVEURS_MLFLOW = ["projet", "personnel"]
f_xlsx = LECTURES / f"grands_modeles_{CAMPAGNE}{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_xlsx, engine="openpyxl") as w:
    TAB_JUSTES.reset_index().to_excel(w, sheet_name="1_croise_justes", index=False)
    CROISE.to_excel(w, sheet_name="2_croise_detail", index=False)
    DUELS.to_excel(w, sheet_name="3_mcnemar", index=False)
    COUTS.to_excel(w, sheet_name="4_couts", index=False)
    for s in SUJETS:
        EV[s].synthese.to_excel(w, sheet_name=f"5_{s}"[:31], index=False)
        L[s].essai.to_excel(w, sheet_name=f"6_essai_{s}"[:31], index=False)
print(f"Excel : {f_xlsx}")


def nom_mlflow(*m):
    return re.sub(r"[^a-z0-9_.-]+", "_", "_".join(str(x) for x in m).lower()).strip("_")[:240]


def git(*args):
    try:
        return subprocess.run(["git", *args], capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return ""


def enregistrer():
    with mlflow.start_run(run_name=f"grands-modeles-{CAMPAGNE}{SUFFIXE}") as run:
        mlflow.set_tags({"etude": "grands-modeles-open-weight", "notebook": "grands_modeles_open_weight",
                         "statut": "exploratoire", "campagne": CAMPAGNE, "sujets": ",".join(SUJETS),
                         "test_rapide": str(TEST_RAPIDE), "git_commit": git("rev-parse", "--short", "HEAD") or "inconnu",
                         "git_branche": git("branch", "--show-current"),
                         "git_code_modifie": "oui" if git("status", "--porcelain") else "non"})
        mlflow.log_params({"consignes": ",".join(CONSIGNES), "candidats": ", ".join(CANDIDATS_EDEN)[:500],
                           "references_internes": ",".join(REFERENCES_INTERNES), "suffixe_eu": SUFFIXE_EU or "non",
                           "plafond_dollars": PLAFOND_DOLLARS, "consignes_de_saisie": True})
        metriques = {"cout_eden_dollars": COMPTEURS.cout}
        for s in SUJETS:
            for _, r_ in EV[s].synthese.iterrows():
                base = nom_mlflow(s, r_["champ"], r_["lecteur"])
                metriques[f"{base}.justes_pct"] = r_["justes_%"]
                metriques[f"{base}.silencieuses_pct"] = r_["silencieuses_%"]
        for _, r_ in DUELS.iterrows():
            metriques[f"mcnemar.{nom_mlflow(r_['sujet'], r_['grand modele'])}.p"] = r_["p"]
        mlflow.log_metrics({k: float(v) for k, v in metriques.items() if pd.notna(v)})
        mlflow.log_artifact(str(f_xlsx))
        for s in SUJETS:
            mlflow.log_text(PAGES[s], f"erreurs/{s}.html")
            mlflow.log_text(L[s].textes.get(CONSIGNES[-1], ""), f"consignes/{s}_{CONSIGNES[-1]}.txt")
            with tempfile.TemporaryDirectory() as d_:
                EV[s].table.to_csv(Path(d_) / "table_ligne_par_ligne.csv", index=False)
                for nom, (_, _, lus) in L[s].lectures.items():
                    lus.to_csv(Path(d_) / f"lectures_{nom_mlflow(nom)}.csv", index=False)
                mlflow.log_artifacts(d_, f"detail/{s}")
        mlflow.log_text("\n".join(PHRASES), "resume.txt")
        return run.info.run_id


# 12. PHRASES POUR LA REUNION (avant l'enregistrement MLflow, qui les garde)
PHRASES = []
meilleur_global = TAB_JUSTES.drop(index="PMQ (saisie humaine)", errors="ignore")["moyenne"].idxmax()
PHRASES.append(f"Sur {len(SUJETS)} sujets ({', '.join(D[s].sujet.titre for s in SUJETS)}), le meilleur modele en "
               f"moyenne est {meilleur_global} ({TAB_JUSTES.loc[meilleur_global, 'moyenne']} % de justes).")
for s in SUJETS:
    dd = DUELS[DUELS.sujet == D[s].sujet.titre]
    gagnants = dd[dd.conclusion == "meilleur que l'interne"]
    PHRASES.append(f"{D[s].sujet.titre} : " + (
        f"{len(gagnants)} grand(s) modele(s) font mieux que notre meilleur interne ({', '.join(gagnants['grand modele'])})."
        if len(gagnants) else "aucun grand modele ne fait significativement mieux que notre meilleur interne."))
if TEST_RAPIDE:
    PHRASES.append("TEST RAPIDE : quelques bulletins seulement, chiffres a ne pas citer.")
for k, p_ in enumerate(PHRASES, 1):
    print(f"{k}. {p_}")

In [ ]:
# 13. MLflow
RUNS = dm.partout(enregistrer, perso="personnel" in SERVEURS_MLFLOW, partage="projet" in SERVEURS_MLFLOW)
print(RUNS)

## Comment lire et conclure

1. **Cellule 7** : le tableau croisé donne, pour chaque modèle, le pourcentage de justes sur chaque sujet (le champ
   principal : nom et prénom tous deux justes, date entière juste, adresse de la voie juste), à côté du PMQ.
2. **Cellule 8** : seul McNemar dit si un grand modèle fait **vraiment** mieux que notre meilleur interne sur les
   mêmes bulletins. « pas d'écart établi » = on ne peut pas conclure qu'il est meilleur.
3. **Cellule 9** : un modèle meilleur mais énorme (des centaines de milliards de paramètres) demande beaucoup de
   cartes graphiques pour être installé : la décision se prend avec la DIAN.
4. **Pages HTML** (cellule 10) : regarder les erreurs qui restent ; « écrit : » montre la lecture avant mise au format.